# Training scClone2DR on the Melanoma or AML cohort with difference features and considering the clones from SCATrEX or the clusters

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from copy import deepcopy
import pickle
import torch
from tqdm import tqdm
import os
import pandas as pd
import plotly.io as pio

import scclone2dr

def training(COHORT, gene_set_collection, clonemode, penalty_l1=3.2, penalty_l2=1.0):
    mode_features = 'metacells_{0}_{1}'.format(gene_set_collection, clonemode)
    if COHORT=='melanoma':
        path_rna = '/data/users/04_share_reanalysis_results/melanoma_2025/02_atypical_removed_preprocessing/{0}/'.format(mode_features)
        path_fastdrug = '/data/users/04_share_reanalysis_results/melanoma_2025/MEL_CNN_abundances_no_plate_effect_correction.csv'
        concentration_DMSO = '100'
        concentration_drug = '5'
        path_info_cohort = None
    elif COHORT=='aml':
        concentration_DMSO = '200'
        concentration_drug = '10'
        path_rna = '/data/users/04_share_reanalysis_results/aml_2025/02_atypical_removed_preprocessing/{0}/'.format(mode_features)
        path_fastdrug = '/data/users/04_share_reanalysis_results/01_aml/AML_PCY_cell_numbers_no_plate_effect_correction.csv'
        path_info_cohort = '/data/users/04_share_reanalysis_results/01_aml/2024-08-15_aml_overview_scRNA.tsv'
    
    datamodule = scclone2dr.data.RealData(path_fastdrug=path_fastdrug, path_rna=path_rna)
    data_ref = datamodule.get_real_data(concentration_DMSO=concentration_DMSO, concentration_drug=concentration_drug)
    
    idxs_train = [i for i in range(int(data_ref['N']))]
    idxs_test = [i for i in range(data_ref['N']) if not(i in idxs_train)]

    data_train, data_test, sample_names_train, sample_names_test = datamodule.get_real_data_split(idxs_train, idxs_test)
    
    from pathlib import Path
    import numpy as np
    from scclone2dr.pipeline import scClone2DRPipeline
    from scclone2dr.trainer import Trainer, GuideType

    # 1) Build the pipeline from the already-loaded real-data module.
    trainer = Trainer(guide_type=GuideType.LOWRANK_MVN, rank=10)
    pipeline = scClone2DRPipeline(
        data_source=datamodule,
        trainer=trainer,
        mode_nu="noise_correction",
        mode_theta="not shared decoupled",
    )

    # Ensure model topology is configured from the dataset metadata.
    pipeline.model.configure(datamodule)

    # 2) Fit on the  already prepared data_train dictionary.
    params_svi = pipeline.fit(
        data=data_train,
        penalty_l1=penalty_l1,
        penalty_l2=penalty_l2,
        lr=0.01,
        n_steps=2000,
    )

    # 3) Save learned parameters.
    ckpt_dir = Path("/data/users/quentin/PACKAGE/checkpoints/")
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = ckpt_dir / f"{mode_features}.npz"
    pipeline.save(ckpt_path)

    # Sample from posterior (Monte Carlo).
    posterior_results = pipeline.sample_posterior(
        data=data_train,
        idxs_sample_eval=idxs_train,
        nb_ites=200,
        dir_save=ckpt_dir,
        sample_names=sample_names_train,
        model_name = mode_features
    )

# Training models

In [ ]:
COHORTS = ['melanoma','aml']
gene_set_collections = ['geneOncoKB','gene','c6','hallmarks', 'c2_pid']
cohort2clonemodes = {'melanoma': ["scatrex",'scatrex_rawcounts_scvi', 'phenograph'], 'aml':['phenograph']}

# Training: melanoma cohort
COHORT = COHORTS[0]
gene_set_collection = gene_set_collections[3]
clonemode = cohort2clonemodes[COHORT][0]
training(COHORT, gene_set_collection, clonemode)

# Training: aml cohort
COHORT = COHORTS[1]
gene_set_collection = gene_set_collections[3]
clonemode = cohort2clonemodes[COHORT][0]
training(COHORT, gene_set_collection, clonemode)